# Laplace vs. Jacobian sensitivity at the MAP

Two different post-hoc analyses you can do once you've found a MAP, and a common Hessian sits underneath both.

- **Laplace approximation** — the Gaussian $\mathcal{N}(x^*, H^{-1})$ built from the curvature of the log posterior at the mode. Captures **posterior uncertainty**.
- **Jacobian sensitivity** — the implicit derivative $dx^*/d\alpha = -H^{-1}\,\partial^2 \log p / \partial x \partial \alpha$ of the MAP with respect to a perturbation $\alpha$ (here, a prior hyperparameter). Captures **how the point estimate moves under model perturbation**.

Same $H$, different geometric objects extracted from it: an **ellipse** (uncertainty) versus an **arrow** (responsiveness).

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects as go
from plotly.subplots import make_subplots

jax.config.update("jax_enable_x64", True)
np.random.seed(0)

## Model: 2D Bayesian logistic regression

We use weights $w \in \mathbb{R}^2$ so the log posterior lives over a 2D plane and can be plotted as an interactive 3D surface.

- Likelihood: $y_i \mid x_i, w \sim \text{Bernoulli}(\sigma(w \cdot x_i))$
- Prior: $w \sim \mathcal{N}(0,\, \alpha^{-1} I)$ — Gaussian with prior precision $\alpha$
- Hyperparameter: $\alpha$. This is the perturbation we will do sensitivity analysis on.

The log posterior (up to a constant) is

$$
L(w; \alpha) \;=\; \sum_i \big[y_i \log \sigma(w \cdot x_i) + (1 - y_i) \log(1 - \sigma(w \cdot x_i))\big] \;-\; \tfrac{\alpha}{2}\,\|w\|^2.
$$

We pick small $N$ on purpose: with little data, the likelihood is asymmetric, so the true posterior is *visibly* non-Gaussian — that is what makes the Laplace approximation fail in the tails.

In [2]:
N = 18
X_np = np.random.randn(N, 2)
true_w = np.array([1.5, -1.2])
probs = 1.0 / (1.0 + np.exp(-X_np @ true_w))
y_np = (np.random.rand(N) < probs).astype(np.float64)

X = jnp.asarray(X_np)
y = jnp.asarray(y_np)
print(f"N = {N},  positives = {int(y_np.sum())},  negatives = {int((1 - y_np).sum())}")

N = 18,  positives = 14,  negatives = 4


## Log posterior, gradient, Hessian — all via autodiff

We define the log posterior once and let JAX differentiate. The negative log posterior is what we minimize for the MAP.

In [3]:
def log_posterior(w, alpha, X, y):
    log_lik = jnp.sum(
        y * jax.nn.log_sigmoid(X @ w) + (1.0 - y) * jax.nn.log_sigmoid(-(X @ w))
    )
    log_prior = -0.5 * alpha * jnp.dot(w, w)
    return log_lik + log_prior

def neg_log_post(w, alpha):
    return -log_posterior(w, alpha, X, y)

ALPHA0 = 1.0  # nominal prior precision; sensitivity is computed at this value

## Find the MAP

BFGS minimization of the negative log posterior. We supply the gradient via JAX so convergence is tight enough that the Hessian we compute next is meaningful.

In [4]:
def find_map(alpha, w0=None):
    if w0 is None:
        w0 = np.zeros(2)
    grad_jit = jax.jit(jax.grad(lambda w: neg_log_post(w, alpha)))
    obj_jit = jax.jit(lambda w: neg_log_post(w, alpha))
    res = minimize(
        lambda w: float(obj_jit(jnp.asarray(w))),
        w0,
        jac=lambda w: np.asarray(grad_jit(jnp.asarray(w))),
        method='BFGS',
        tol=1e-12,
    )
    return np.asarray(res.x)

w_star = find_map(ALPHA0)
print(f"MAP estimate w*       = {w_star}")
print(f"True w (data-gen)     = {true_w}")

MAP estimate w*       = [ 0.78709625 -0.74441239]
True w (data-gen)     = [ 1.5 -1.2]


## The Hessian at the MAP and its eigendecomposition

$H = \nabla^2_w(-\log p)\big|_{w^*}$ is positive definite because the MAP is a local minimum of the negative log posterior. The Laplace covariance is $\Sigma = H^{-1}$.

Below we compute $H$, $H^{-1}$, and the eigendecomposition of $H^{-1}$.

- **Eigenvectors of $H^{-1}$** → the principal axes of the uncertainty ellipse.
- **Eigenvalues of $H^{-1}$** → the variances along those axes (so $\sqrt{\lambda_i}$ is the σ along axis $i$).

Print the decomposition next to the picture so you can match numbers to geometry.

In [5]:
hess_fn = jax.jit(jax.hessian(lambda w: neg_log_post(w, ALPHA0)))
H = np.asarray(hess_fn(jnp.asarray(w_star)))

H_inv = np.linalg.inv(H)
eigvals, eigvecs = np.linalg.eigh(H_inv)  # ascending eigenvalues

print("H (curvature of negative log posterior at MAP):")
print(np.round(H, 4))
print("\nH^-1 (Laplace covariance):")
print(np.round(H_inv, 4))
print("\nEigendecomposition of H^-1:")
for i, (lam, v) in enumerate(zip(eigvals, eigvecs.T)):
    print(f"  axis {i}:  variance lambda = {lam:.4f}   sigma = {np.sqrt(lam):.4f}   direction = {np.round(v, 3)}")

H (curvature of negative log posterior at MAP):
[[4.5088 0.9168]
 [0.9168 4.4742]]

H^-1 (Laplace covariance):
[[ 0.2314 -0.0474]
 [-0.0474  0.2332]]

Eigendecomposition of H^-1:
  axis 0:  variance lambda = 0.1849   sigma = 0.4300   direction = [-0.714 -0.7  ]
  axis 1:  variance lambda = 0.2798   sigma = 0.5289   direction = [-0.7    0.714]


## The log posterior surface

Drag to rotate. The red dot is the MAP. This is the actual log posterior — not the Laplace approximation yet.

In [6]:
GRID = 90
PAD = 2.6
w1_range = np.linspace(w_star[0] - PAD, w_star[0] + PAD, GRID)
w2_range = np.linspace(w_star[1] - PAD, w_star[1] + PAD, GRID)
W1, W2 = np.meshgrid(w1_range, w2_range)

log_post_at_alpha0 = jax.jit(lambda w: log_posterior(w, ALPHA0, X, y))
log_post_vec = jax.jit(jax.vmap(log_post_at_alpha0))

W_grid = jnp.stack([W1.ravel(), W2.ravel()], axis=-1)
Z = np.asarray(log_post_vec(W_grid)).reshape(W1.shape)
z_at_map = float(log_post_at_alpha0(jnp.asarray(w_star)))

fig = go.Figure([
    go.Surface(x=W1, y=W2, z=Z, colorscale='Viridis', opacity=0.9, showscale=False),
    go.Scatter3d(
        x=[w_star[0]], y=[w_star[1]], z=[z_at_map],
        mode='markers',
        marker=dict(size=6, color='red'),
        name='MAP',
    ),
])
fig.update_layout(
    title='Log posterior over w. Drag to rotate; scroll to zoom.',
    scene=dict(xaxis_title='w1', yaxis_title='w2', zaxis_title='log p(w | y)'),
    width=820, height=620, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## Method 1: Laplace approximation

Replace $p(w \mid y)$ with $\mathcal{N}(w^*, H^{-1})$. Its log density is the second-order Taylor expansion of the log posterior at the mode:

$$
\log q(w) = -\tfrac{1}{2}(w - w^*)^\top H (w - w^*) + \text{const}.
$$

Side-by-side: true log posterior contours vs. quadratic approximation, on the *same* contour levels. They agree near the MAP — by construction. Watch for differences in the tails.

**Takeaway.** Laplace replaces the full landscape with a quadratic bowl. The quadratic bowl is a perfect ellipsoid; the actual posterior is not.

In [7]:
delta = np.stack([W1 - w_star[0], W2 - w_star[1]], axis=-1)
quad = np.einsum('...i,ij,...j->...', delta, H, delta)
Z_laplace = z_at_map - 0.5 * quad

contour_kw = dict(
    start=Z.min(),
    end=z_at_map,
    size=(z_at_map - Z.min()) / 15,
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('True log posterior', 'Laplace (quadratic) approximation'),
)
fig.add_trace(
    go.Contour(x=w1_range, y=w2_range, z=Z, colorscale='Viridis',
               contours=contour_kw, showscale=False),
    row=1, col=1,
)
fig.add_trace(
    go.Contour(x=w1_range, y=w2_range, z=Z_laplace, colorscale='Viridis',
               contours=contour_kw, showscale=False),
    row=1, col=2,
)
for col in (1, 2):
    fig.add_trace(
        go.Scatter(x=[w_star[0]], y=[w_star[1]], mode='markers',
                   marker=dict(size=10, color='red'),
                   showlegend=(col == 1), name='MAP' if col == 1 else None),
        row=1, col=col,
    )
fig.update_xaxes(title_text='w1')
fig.update_yaxes(title_text='w2')
fig.update_layout(width=950, height=460,
                  title='Same contour levels. Identical near MAP, different in the tails.')
fig.show()

## Where the Laplace approximation breaks down

Laplace gives **perfectly elliptical** level sets. The true log posterior generally does not — especially with small $N$ or class imbalance, where the likelihood is asymmetric. The plot below shows $(\text{true log posterior}) - (\text{Laplace})$.

- **Red regions** ($> 0$): the truth is heavier than Laplace (Laplace underestimates probability mass here).
- **Blue regions** ($< 0$): Laplace is heavier than the truth (Laplace overestimates).

Zero by construction at the MAP and to first order around it. Errors grow as you move out into the tails.

**Takeaway.** Laplace is a *local* approximation. Reliable for posterior moments and concentration arguments around the mode; misleading in the tails — exactly where rare events live.

In [8]:
diff = Z - Z_laplace
absmax = float(np.max(np.abs(diff)))

fig = go.Figure([
    go.Contour(x=w1_range, y=w2_range, z=diff,
               colorscale='RdBu_r', zmid=0, zmin=-absmax, zmax=absmax,
               colorbar=dict(title='log p − Laplace')),
    go.Scatter(x=[w_star[0]], y=[w_star[1]], mode='markers',
               marker=dict(size=10, color='black'), name='MAP'),
])
fig.update_layout(width=720, height=520,
                  title='Approximation error: true log posterior minus Laplace',
                  xaxis_title='w1', yaxis_title='w2')
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

## The Laplace ellipse — eigendecomposition made visual

The 1-σ contour of $\mathcal{N}(w^*, H^{-1})$ is the set $\{w : (w - w^*)^\top H (w - w^*) = 1\}$.

Decompose $H^{-1} = V \Lambda V^\top$. Then:

- The **principal axes** of the ellipse are the columns of $V$ (eigenvectors of $H^{-1}$).
- The **half-lengths** along those axes are $\sqrt{\lambda_i}$.

Below, the red curve is the 1-σ ellipse; the dashed curve is 2-σ. The black segments through the MAP are the eigenvector axes scaled by $\sqrt{\lambda}$. Compare these to the printed eigendecomposition above — same numbers, now as geometry.

In [9]:
theta = np.linspace(0.0, 2.0 * np.pi, 240)
unit_circle = np.stack([np.cos(theta), np.sin(theta)], axis=0)

# H^-1 = V Lambda V^T  →  H_inv_sqrt = V Lambda^{1/2} V^T
H_inv_sqrt = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
ellipse_1 = w_star[:, None] + H_inv_sqrt @ unit_circle
ellipse_2 = w_star[:, None] + 2.0 * H_inv_sqrt @ unit_circle

fig = go.Figure([
    go.Contour(x=w1_range, y=w2_range, z=Z, colorscale='Viridis', showscale=False,
               contours=dict(start=Z.min(), end=z_at_map,
                             size=(z_at_map - Z.min()) / 15)),
    go.Scatter(x=ellipse_1[0], y=ellipse_1[1], mode='lines',
               line=dict(color='crimson', width=3), name='Laplace 1σ'),
    go.Scatter(x=ellipse_2[0], y=ellipse_2[1], mode='lines',
               line=dict(color='crimson', width=2, dash='dash'), name='Laplace 2σ'),
])

for i, lam in enumerate(eigvals):
    half = np.sqrt(lam) * eigvecs[:, i]
    fig.add_trace(go.Scatter(
        x=[w_star[0] - half[0], w_star[0] + half[0]],
        y=[w_star[1] - half[1], w_star[1] + half[1]],
        mode='lines+markers',
        line=dict(color='black', width=2),
        marker=dict(size=4, color='black'),
        name=f'eigvec axis {i} (sigma = {np.sqrt(lam):.3f})',
    ))

fig.add_trace(go.Scatter(x=[w_star[0]], y=[w_star[1]], mode='markers',
                         marker=dict(size=10, color='red'), name='MAP'))

fig.update_layout(width=720, height=620,
                  title='Laplace ellipse, 1σ and 2σ, with eigenvector axes',
                  xaxis_title='w1', yaxis_title='w2')
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

## Method 2: Jacobian sensitivity via implicit differentiation

A different question. How does the **point estimate** $w^*$ move when we perturb a hyperparameter $\alpha$?

Let $f(w, \alpha) = -\log p(w \mid y; \alpha)$. The MAP $w^*(\alpha)$ satisfies the first-order condition $\nabla_w f(w^*(\alpha), \alpha) = 0$. Differentiate both sides w.r.t. $\alpha$:

$$
H \cdot \frac{dw^*}{d\alpha} \;+\; \nabla^2_{w \alpha} f \;=\; 0
\quad\Longrightarrow\quad
\boxed{\;\frac{dw^*}{d\alpha} \;=\; -H^{-1}\, \nabla^2_{w \alpha} f\;}
$$

The **same $H$** appears here as in Laplace — but now we *apply* $H^{-1}$ as a linear operator to a specific vector ($\nabla^2_{w \alpha} f$), instead of visualizing it as an ellipse. The product $H^{-1} v$ is one direction in $w$-space: how the optimum moves under a tiny change in $\alpha$.

For our prior $-\tfrac{\alpha}{2}\|w\|^2$, the cross-derivative is

$$
\nabla^2_{w \alpha} f \;=\; \nabla_w\!\left(\tfrac{1}{2}\|w\|^2\right) \;=\; w.
$$

So at the MAP, $\nabla^2_{w \alpha} f \big|_{w^*} = w^*$, and the formula reduces to

$$
\frac{dw^*}{d\alpha} \;=\; -H^{-1}\, w^*.
$$

**Sign-check intuition.** $H^{-1}$ is positive definite, so $-H^{-1} w^*$ points roughly *toward the origin* relative to $w^*$. That's exactly what a stronger prior (larger $\alpha$) should do: shrink the MAP toward zero. Sign of the formula matches the physics.

In [10]:
# Cross-derivative ∂²f / ∂w ∂α at MAP — autodiff
d2_dw_da = jax.jit(jax.grad(jax.grad(neg_log_post, argnums=1), argnums=0))
cross = np.asarray(d2_dw_da(jnp.asarray(w_star), ALPHA0))

print(f"∂²f / ∂w ∂α  at MAP   = {cross}")
print(f"w* (should match)      = {w_star}")
print(f"max abs error          = {np.max(np.abs(cross - w_star)):.2e}")

dw_dalpha_implicit = -np.linalg.solve(H, cross)
print(f"\nAnalytic dw*/dα via implicit differentiation: {dw_dalpha_implicit}")

∂²f / ∂w ∂α  at MAP   = [ 0.78709625 -0.74441239]
w* (should match)      = [ 0.78709625 -0.74441239]
max abs error          = 0.00e+00

Analytic dw*/dα via implicit differentiation: [-0.21746133  0.21094018]


## Validation: implicit derivative vs. finite-difference re-optimization

This is half the pedagogical point. The implicit-differentiation formula is an algebraic identity, but it's also a **prediction** about what happens when you re-run the optimizer at a perturbed $\alpha$. Confirm the prediction by central-differencing two re-optimized MAPs.

If they agree, the formula isn't bookkeeping — it's making a real claim, and the claim holds.

In [11]:
EPS = 1e-3
w_plus  = find_map(ALPHA0 + EPS, w0=w_star)
w_minus = find_map(ALPHA0 - EPS, w0=w_star)
dw_dalpha_fd = (w_plus - w_minus) / (2 * EPS)

print(f"Implicit-diff dw*/dα   = {dw_dalpha_implicit}")
print(f"Finite-diff   dw*/dα   = {dw_dalpha_fd}")
print(f"Max abs difference     = {np.max(np.abs(dw_dalpha_implicit - dw_dalpha_fd)):.2e}")
print()
print("→ Agreement to many digits. The implicit-differentiation formula predicts")
print("  exactly where re-optimization moves the MAP. Same H that gave the Laplace")
print("  covariance also gives this prediction.")

Implicit-diff dw*/dα   = [-0.21746133  0.21094018]
Finite-diff   dw*/dα   = [-0.21746138  0.21094022]
Max abs difference     = 4.57e-08

→ Agreement to many digits. The implicit-differentiation formula predicts
  exactly where re-optimization moves the MAP. Same H that gave the Laplace
  covariance also gives this prediction.


## The MAP traces a path as $\alpha$ varies

To put the local Jacobian in context, sweep $\alpha$ from very weak (~0.1, almost flat prior) to strong (5, tight shrinkage), and plot the path $w^*(\alpha)$. The Jacobian arrow at $\alpha = 1$ is the **local tangent** of this path.

You'll see the path bend as $\alpha$ grows — the MAP shrinks toward the origin. The arrow only gets the local direction right, but it gets it right *exactly* (matching the finite-difference check above).

In [12]:
alpha_sweep = np.linspace(0.1, 5.0, 40)
map_path = np.array([find_map(a) for a in alpha_sweep])

ARROW = 0.5  # cosmetic arrow scale (units: α)
arrow_end = w_star + ARROW * dw_dalpha_implicit

fig = go.Figure([
    go.Contour(x=w1_range, y=w2_range, z=Z, colorscale='Viridis', showscale=False, opacity=0.5,
               contours=dict(start=Z.min(), end=z_at_map,
                             size=(z_at_map - Z.min()) / 15)),
    go.Scatter(
        x=map_path[:, 0], y=map_path[:, 1],
        mode='lines+markers',
        marker=dict(size=5, color=alpha_sweep, colorscale='Plasma',
                    showscale=True, colorbar=dict(title='α')),
        line=dict(color='gray', width=1.5),
        name='MAP path w*(α)',
    ),
    go.Scatter(x=[w_star[0]], y=[w_star[1]], mode='markers',
               marker=dict(size=12, color='red'), name='w* at α = 1'),
    go.Scatter(x=[w_star[0], arrow_end[0]], y=[w_star[1], arrow_end[1]],
               mode='lines', line=dict(color='red', width=4),
               name='dw*/dα × 0.5  (analytic tangent)'),
])

fig.add_annotation(
    x=arrow_end[0], y=arrow_end[1],
    ax=w_star[0], ay=w_star[1],
    xref='x', yref='y', axref='x', ayref='y',
    showarrow=True, arrowhead=3, arrowwidth=4, arrowcolor='red',
)

fig.update_layout(width=720, height=620,
                  title='MAP path w*(α). Red arrow = analytic tangent at α = 1.',
                  xaxis_title='w1', yaxis_title='w2')
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

## The conceptual punchline: ellipse vs. arrow at the same point

Same Hessian $H$. Same MAP $w^*$. Two completely different geometric objects extracted from $H$:

- **Blue ellipse** — $H^{-1}$ as a *quadratic form* — the **uncertainty** around $w^*$. A region. No preferred direction along its principal axes.
- **Red arrow** — $H^{-1}$ as a *linear operator applied to $\nabla^2_{w \alpha} f$* — the **responsiveness** of $w^*$ to a change in $\alpha$. A single direction. With sign and magnitude.

The ellipse asks: *if the data spoke a little less loudly, how spread out would my belief be?*

The arrow asks: *if my model assumption $\alpha$ were slightly different, where would my point estimate move?*

Different questions. Same Hessian. Different extractions.

In [13]:
ARROW = 0.5
arrow_end = w_star + ARROW * dw_dalpha_implicit

fig = go.Figure([
    go.Contour(x=w1_range, y=w2_range, z=Z, colorscale='Viridis', showscale=False, opacity=0.4,
               contours=dict(start=Z.min(), end=z_at_map,
                             size=(z_at_map - Z.min()) / 15)),
    go.Scatter(x=ellipse_1[0], y=ellipse_1[1], mode='lines',
               line=dict(color='royalblue', width=3),
               name='Laplace ellipse  (1σ uncertainty)'),
])

# eigenvector axes (cosmetic, dotted)
for i, lam in enumerate(eigvals):
    half = np.sqrt(lam) * eigvecs[:, i]
    fig.add_trace(go.Scatter(
        x=[w_star[0] - half[0], w_star[0] + half[0]],
        y=[w_star[1] - half[1], w_star[1] + half[1]],
        mode='lines',
        line=dict(color='royalblue', width=1, dash='dot'),
        showlegend=(i == 0),
        name='ellipse principal axes' if i == 0 else None,
    ))

# Jacobian arrow
fig.add_trace(go.Scatter(
    x=[w_star[0], arrow_end[0]], y=[w_star[1], arrow_end[1]],
    mode='lines',
    line=dict(color='crimson', width=4),
    name='dw*/dα × 0.5  (sensitivity arrow)',
))
fig.add_annotation(
    x=arrow_end[0], y=arrow_end[1],
    ax=w_star[0], ay=w_star[1],
    xref='x', yref='y', axref='x', ayref='y',
    showarrow=True, arrowhead=3, arrowwidth=4, arrowcolor='crimson',
)

fig.add_trace(go.Scatter(x=[w_star[0]], y=[w_star[1]], mode='markers',
                         marker=dict(size=14, color='black'), name='MAP w*'))

fig.update_layout(
    width=820, height=720,
    title='Same H, two different objects extracted from it',
    xaxis_title='w1', yaxis_title='w2',
    legend=dict(yanchor='top', y=0.98, xanchor='left', x=0.02,
                bgcolor='rgba(255,255,255,0.85)'),
)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

## Takeaways

1. **One Hessian, two extractions.**
   - $H^{-1}$ as a *quadratic form* → the uncertainty ellipse (Laplace).
   - $H^{-1}$ as a *linear operator* applied to $\nabla^2_{w \alpha} f$ → the sensitivity arrow (implicit differentiation).

2. **Laplace is good locally and bad in the tails.** The error map shows it: identical to the truth at the MAP and to first order around it; visibly off as you move out. Trust Laplace for posterior moments and concentration arguments; do not trust it for rare-event probabilities.

3. **Implicit differentiation is verifiable.** When the analytic $-H^{-1} \nabla^2_{w \alpha} f$ matches finite-difference re-optimization to many digits, you've confirmed the formula is making a real claim, not just shuffling symbols. Always do this validation — it's the half of the story that's easy to skip.

4. **Eigendecomposition connects algebra to picture.**
   - Eigenvectors of $H^{-1}$ → principal axes of the ellipse.
   - Eigenvalues of $H^{-1}$ → variances along those axes ($\sqrt{\lambda}$ is the σ).
   - A direction with small posterior variance (small $\lambda$ in $H^{-1}$, equivalently large $H$ eigenvalue) is one the data has pinned tightly; both the ellipse half-length *and* the projection of the sensitivity arrow onto that direction are correspondingly small.

5. **What's *not* captured by either method.** Both inherit a local-quadratic story. Genuinely non-Gaussian posteriors (multimodality, heavy tails, banana shapes) need MCMC or a more flexible variational family. Sensitivity at a single MAP also misses what happens when the perturbation is large enough that $w^*(\alpha)$ jumps to a different basin — the linear formula predicts smooth motion only as long as the optimizer stays on the same branch.